# ACML: Adaptive Categorical Meta-Learning

This notebook demonstrates the ACML framework which implements:
- **Category-theoretic neural architectures**
- **Natural gradient descent** using Fisher information
- **Sheaf-based federated learning**

## Mathematical Foundation

ACML treats neural networks as functors between enriched categories:
```
N: Arch → Vect_R
```

Parameters are updated using the Fisher metric:
```
θ_{t+1} = θ_t - α F^{-1}(θ) ∇L(θ)
```

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

from acml import (
    EnrichedCategory,
    FunctorialLayer,
    FisherInformationEstimator,
    NaturalGradientOptimizer,
    CategoricalMetaLearner,
    ACMLConfig,
)

print("✅ ACML imported successfully!")

## 1. Enriched Categories

Categories enriched with metric, inner product, or probabilistic structure.

In [ ]:
# Create enriched categories
metric_cat = EnrichedCategory('metric')
inner_product_cat = EnrichedCategory('inner_product')
prob_cat = EnrichedCategory('probability')

# Test data
x = torch.randn(5, 10)
y = torch.randn(5, 10)

# Compute distances/similarities
metric_dist = metric_cat.hom(x, y)
inner_sim = inner_product_cat.hom(x, y)
prob_div = prob_cat.hom(x, y)

print(f"Metric distance shape: {metric_dist.shape}")
print(f"Inner product similarity shape: {inner_sim.shape}")
print(f"Probability divergence shape: {prob_div.shape}")

## 2. Functorial Neural Networks

Neural network layers as functors between categories.

In [ ]:
# Build a functorial network
enrichment = EnrichedCategory('metric')

layers = [
    FunctorialLayer(784, 256, enrichment, activation=nn.ReLU()),
    FunctorialLayer(256, 128, enrichment, activation=nn.ReLU()),
    FunctorialLayer(128, 10, enrichment, activation=None)
]

functorial_net = nn.Sequential(*layers)

# Test forward pass
test_input = torch.randn(32, 784)
output = functorial_net(test_input)

print(f"Input shape: {test_input.shape}")
print(f"Output shape: {output.shape}")
print(f"\nFunctor structure:")
for i, layer in enumerate(layers):
    print(f"Layer {i}: {layer.get_functor_structure()}")

## 3. Fisher Information & Natural Gradients

Estimate Fisher information matrix and compute natural gradients.

In [ ]:
# Create model and estimator
model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 5)
)

fisher_estimator = FisherInformationEstimator(
    damping=1e-4,
    num_samples=50
)

# Generate data
data = torch.randn(100, 10)

# Estimate Fisher information
fisher_matrix = fisher_estimator.estimate(model, data, use_empirical_fisher=True)

print(f"Fisher matrix shape: {fisher_matrix.shape}")
print(f"Fisher matrix symmetric: {torch.allclose(fisher_matrix, fisher_matrix.T)}")

# Compute natural gradient
gradient = torch.randn(fisher_matrix.shape[0])
natural_gradient = fisher_estimator.compute_natural_gradient(fisher_matrix, gradient)

print(f"\nNatural gradient computed: {natural_gradient.shape}")

## 4. Meta-Learning Workflow

Complete meta-learning example with task adaptation.

In [ ]:
# Configure ACML
config = ACMLConfig(
    meta_lr=1e-3,
    inner_lr=1e-2,
    inner_steps=5,
    meta_batch_size=16,
    use_natural_gradient=False,  # Use standard for demo
    encoder_dims=[784, 256, 128, 64],
    decoder_dims=[64, 128, 10]
)

# Create meta-learner
meta_learner = CategoricalMetaLearner(config)

print(f"Meta-learner initialized")
print(f"Encoder: {config.encoder_dims}")
print(f"Decoder: {config.decoder_dims}")

In [ ]:
# Generate synthetic few-shot learning tasks
def generate_task(n_support=5, n_query=15):
    """Generate a synthetic classification task."""
    support_x = torch.randn(n_support, 784)
    support_y = torch.randint(0, 10, (n_support,))
    query_x = torch.randn(n_query, 784)
    query_y = torch.randint(0, 10, (n_query,))
    return {
        'support_x': support_x,
        'support_y': support_y,
        'query_x': query_x,
        'query_y': query_y
    }

# Create task batch
task_batch = [generate_task() for _ in range(4)]

print(f"Generated {len(task_batch)} tasks")
print(f"Task 0 support shape: {task_batch[0]['support_x'].shape}")
print(f"Task 0 query shape: {task_batch[0]['query_x'].shape}")

In [ ]:
# Simulate meta-training
losses = []

for iteration in range(20):
    # Generate new task batch
    task_batch = [generate_task() for _ in range(4)]
    
    # Meta-train step
    try:
        loss = meta_learner.meta_train_step(task_batch)
        losses.append(loss)
        
        if iteration % 5 == 0:
            print(f"Iteration {iteration}: Loss = {loss:.4f}")
    except Exception as e:
        print(f"Error at iteration {iteration}: {e}")
        break

print(f"\nTraining complete! Final loss: {losses[-1]:.4f}")

In [ ]:
# Plot training curve
plt.figure(figsize=(10, 6))
plt.plot(losses)
plt.xlabel('Iteration')
plt.ylabel('Meta-Loss')
plt.title('ACML Meta-Training Progress')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Comparison: Standard vs Natural Gradients

Visualize the difference between standard and natural gradient updates.

In [ ]:
# Compare optimization trajectories
def rosenbrock(x, y):
    """Rosenbrock function for optimization demo."""
    return (1 - x)**2 + 100 * (y - x**2)**2

# Create grid
x = np.linspace(-2, 2, 100)
y = np.linspace(-1, 3, 100)
X, Y = np.meshgrid(x, y)
Z = rosenbrock(X, Y)

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.contour(X, Y, Z, levels=np.logspace(0, 3, 20), cmap='viridis')
plt.colorbar(label='Loss')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Loss Landscape (Rosenbrock)')

plt.subplot(1, 2, 2)
plt.contour(X, Y, Z, levels=np.logspace(0, 3, 20), cmap='viridis')
plt.colorbar(label='Loss')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Optimization Trajectories')
plt.plot([0], [0], 'r*', markersize=15, label='Global Minimum')
plt.legend()

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

1. **Enriched Categories** - Mathematical foundation for neural architectures
2. **Functorial Networks** - Neural layers as category morphisms
3. **Fisher Information** - Riemannian metric for parameter space
4. **Natural Gradients** - Geodesic optimization
5. **Meta-Learning** - Learning to learn across tasks

### Key Takeaways

- ACML provides principled meta-learning via category theory
- Natural gradients account for parameter space geometry
- Few-shot learning enables rapid adaptation to new tasks
- Sheaf-theoretic aggregation for distributed learning